# <center><span style="color:#336699">PYTHON + QGIS - PARTE 2</span></center>
<hr style="border:2px solid #0077b9;">

<br/>

<div style="text-align: center;font-size: 150%;">
    Grupo 7 - Seminário Python + Qgis<br/>
</div>

<br/>

<div style="text-align: center;font-size: 90%;">
    Carla Almeida <sup><a href=""></sup>,     Tayná Florentino <sup><a href=""></i></a></sup>
    <br/><br/>
     Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Atualização: 18 de Maio de 2026
</div>

<br/>

<div style="text-align: justify;  margin-left: 25%; margin-right: 25%;">
    <b>Resumo.</b> Demonstrar como utilizar a API nativa do PyQGIS dentro do Jupyter Notebook para realização de processos como: criação de camadas, realçar o  contraste e a estilização de imagens de satélite diretamente na interface do QGIS, sem a necessidade de cliques manuais ou uso de bibliotecas externas de matrizes. 
</div>

<br/>

##  Importações da API do PyQGIS - Developer Cookbook (Capítulo 5 - Dados Raster).

Importações classes da API  QGIS . Essas classes permite executar algumas funções como : 

 QgsProject: É possível adicionar ou remover camadas.

QgsSingleBandGrayRenderer:  Faz  a renderização do  raster em tons de cinza (ótimo para relevos sombreados ou imagens pancromáticas).

QgsSingleBandColorDataRenderer: Usado quando o seu raster já possui uma tabela de cores embutida direto no arquivo (como alguns mapas temáticos).

QgsSingleBandPseudoColorRenderer: É o renderizador para "falsas cores". Ele pega um raster de uma única banda (como temperatura ou altitude) e aplica um gradiente de cores nele.


In [2]:
import os
from qgis.core import (
    QgsProject,
    QgsRasterLayer,
    QgsContrastEnhancement,
    QgsRasterMinMaxOrigin,
    QgsSingleBandPseudoColorRenderer,
    QgsRasterShader,
    QgsColorRampShader,
    QgsMultiBandColorRenderer
)
from qgis.PyQt.QtGui import QColor

print("API do PyQGIS carregada com Sucesso!")

ModuleNotFoundError: No module named 'qgis'

## Configurando o path das imagens -  Developer Cookbook (Capítulo 3.2 - Camadas Matriciais)

Definição do diretório local onde as bandas (.tif) do CBERS-4A estão armazenadas. Faremos o carregamento da Banda 7 (Filtro Vermelho) para  camadas do QGIS.

In [ ]:
# Acessando arquivos raster, no caso usando a biblioteca GDAL. 

# Definindo a pasta do repositório local
pasta_imagens = os.path.expanduser("~/Documents/Seminario_Prog/cbers_4a_mux")

# Caminho específico da Banda 7
caminho_b7 = os.path.join(pasta_imagens, "CBERS_4A_MUX_20260212_194_138_L2_BAND7.tif")


In [ ]:
#  utilzando o  comando iface.addRasterlayer para criar  uma nova camada e a adiciona ao projeto atual (Podemos visualizar na lista de camadas) 

rlayer = iface.addRasterLayer(caminho_b7, "CBERS_Banda_7_Original")

if rlayer.isValid():
    print("Camada raster carregada com sucesso na interface do QGIS!")
else:
    print("Erro ao carregar a camada. Verifique o caminho do arquivo.")

## Criando a Paleta de Cores Fixa (Dicionário de Cores) - 5.2.1 Rasters com Banda Única

In [ ]:
# 3. Instanciando o Shader de Cores do QGIS
fcn = QgsColorRampShader()
fcn.setColorRampType(QgsColorRampShader.Interpolated)

# 4. FIXANDO OS LIMITES: Mapeando os tons diretamente nos valores dos pixels (8 bits)
# Criamos uma rampa que vai do Verde (pixel 0) ao Vermelho (pixel 255) passando pelo Amarelo
lst = [ 
    QgsColorRampShader.ColorRampItem(0, QColor(0, 255, 0)),      # Tons escuros -> Verde
    QgsColorRampShader.ColorRampItem(127, QColor(255, 255, 0)),  # Tons médios -> Amarelo
    QgsColorRampShader.ColorRampItem(255, QColor(255, 0, 0))     # Tons claros  -> Vermelho
]

# 5. Adiciona a lista de cores dentro do configurador
fcn.setColorRampItemList(lst)

print("Paleta de cores estática (Verde -> Amarelo -> Vermelho) configurada!")

## Construindo o Motor de Sombreamento (Shader)
Aqui o QGIS compila as regras de cores que você criou e gera o renderizador raster nativo.


In [ ]:
# 6. Constrói o Shader do Raster usando as nossas regras

shader = QgsRasterShader()
shader.setRasterShaderFunction(fcn)

# 7. Cria o renderizador de Pseudocor Banda Simples para a Banda 1
renderizador = QgsSingleBandPseudoColorRenderer(rlayer_cor.dataProvider(), 1, shader)

print("Renderizador de pseudocor construído com sucesso.")


## Aplicando o Visual e Atualizando a Interface

o rodar essa última célula, o visual preto e branco antigo some e a imagem colorida aparece na tela.


In [ ]:

# 8. Substitui o renderizador antigo (tons de cinza) pelo nosso colorido
rlayer_cor.setRenderer(renderizador)

# 9. Força o QGIS a redesenhar a tela com o novo mapa de cores
rlayer_cor.triggerRepaint()

print("Mágica feita! Pseudocor aplicada direto na interface sem usar estatísticas.")

# <span style="color:#336699">Referências Bibliográficas </span>
<hr style="border:2px solid #0077b9;">

INPE - Instituto Nacional de Pesquisas Espaciais. Catálogo de Imagens do Satélite CBERS-4A (Sensor MUX). Dados obtidos em Maio de 2026. Disponível em: https://www.dgi.inpe.br/.

INPE - Instituto Nacional de Pesquisas Espaciais. Portal de Dados Geoespaciais do INPE (v.3.0). Av. dos Astronautas, 1.758, São José dos Campos - SP, 2025. Disponível em: https://data.inpe.br/dados/satelite-cbers-4a/.

QGIS Project. PyQGIS Developer Cookbook (Versão 3.44). Documentação oficial do desenvolvedor QGIS, 2026. Disponível em: https://docs.qgis.org/3.44/pdf/pt_BR/QGIS-3.44-PyQGISDeveloperCookbook-pt_BR.pdf.

AD33 Geotecnologias. CBERS-04A: Guia rápido das câmeras e download das imagens. Disponível em: https://www.ad33.com.br/post/cbers-04a-guia-r%C3%A1pido-das-c%C3%A2meras-e-download-das-imagens.

Bibliotecas de Processamento Digital de Imagens (PDI)

Gillies, S. et al. Rasterio: geospatial raster I/O for Python programmers, 2013. Disponível em: https://github.com/rasterio/rasterio.